# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [8]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


# Example selection and encoding for 'HomePlanet'
selected_features = spaceship[['Age', 'RoomService', 'FoodCourt', 'HomePlanet']]  # Adjust columns as needed

# Encoding categorical 'HomePlanet' into dummy variables
selected_features = pd.get_dummies(selected_features, columns=['HomePlanet'], drop_first=True)

# Check data to ensure no non-numeric types
print(selected_features.head())

# Split the data (assuming 'Transported' is your target variable)
X = selected_features
y = spaceship['Transported']  # Ensure this column is numeric (e.g., 0/1 or convert boolean)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize numeric features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Check scaled data
print("Scaled training features:\n", X_train_scaled[:5])

    Age  RoomService  FoodCourt  HomePlanet_Europa  HomePlanet_Mars
0  39.0          0.0        0.0               True            False
1  24.0        109.0        9.0              False            False
2  58.0         43.0     3576.0               True            False
3  33.0          0.0     1283.0               True            False
4  16.0        303.0       70.0              False            False
Scaled training features:
 [[-0.05732596 -0.33018094 -0.25510852 -0.56926885 -0.50049427]
 [-0.81881735 -0.33018094  0.46884733 -0.56926885 -0.50049427]
 [-0.05732596 -0.33018094 -0.29003621  1.75663924 -0.50049427]
 [-0.61113788         nan -0.28876611 -0.56926885  1.99802487]
 [ 0.49648595 -0.33018094 -0.29003621  1.75663924 -0.50049427]]


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [10]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import randint
from sklearn.metrics import accuracy_score, classification_report

# Example best model from previous runs (e.g., RandomForestClassifier)
best_model = RandomForestClassifier()

# Define a more concise hyperparameter space
param_dist = {
    'n_estimators': randint(50, 150),
    'max_depth': [10, 20, None],
    'min_samples_split': randint(2, 11),
    'min_samples_leaf': randint(1, 5),
    'max_features': ['sqrt', 'log2']
}

# Set up Randomized Search
random_search = RandomizedSearchCV(estimator=best_model, param_distributions=param_dist, n_iter=30,
                                   cv=3, n_jobs=-1, verbose=2, scoring='accuracy', random_state=42)

# Fit the random search to the data
random_search.fit(X_train_scaled, y_train)

# Get the best parameters
best_params = random_search.best_params_
print("Best hyperparameters found:", best_params)

# Train a final model with the best parameters
optimized_model = RandomForestClassifier(**best_params)
optimized_model.fit(X_train_scaled, y_train)

# Predict and evaluate on the test set
y_pred = optimized_model.predict(X_test_scaled)

# Evaluate performance
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Optimized Model Accuracy: {accuracy:.2f}")
print("\nClassification Report:")
print(report)

Fitting 3 folds for each of 30 candidates, totalling 90 fits
Best hyperparameters found: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 4, 'n_estimators': 136}
Optimized Model Accuracy: 0.72

Classification Report:
              precision    recall  f1-score   support

       False       0.72      0.71      0.71       861
        True       0.72      0.73      0.72       878

    accuracy                           0.72      1739
   macro avg       0.72      0.72      0.72      1739
weighted avg       0.72      0.72      0.72      1739



**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [11]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Choose the classifier
rf_model = RandomForestClassifier(random_state=42)

# Define a smaller, more manageable hyperparameter grid
param_grid = {
    'n_estimators': [50, 100],  # Fewer trees reduce the complexity
    'max_depth': [10, 20, None],  # Limiting maximum depth can save compute
    'min_samples_split': [2, 5, 10],  # Try a few variations for split counts
    'min_samples_leaf': [1, 2],  # Balanced division gives options but with limited count
    'max_features': ['sqrt', 'log2']  # A couple of methods for selecting features can provide insights
}

# Setup Grid Search
grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2, scoring='accuracy')

# Fit the Grid Search to the data
grid_search.fit(X_train_scaled, y_train)

# Retrieve the best parameters and results
best_params = grid_search.best_params_
print("Best hyperparameters found:", best_params)

# Train the final model with the best parameters
optimized_model = RandomForestClassifier(**best_params, random_state=42)
optimized_model.fit(X_train_scaled, y_train)

# Evaluate the model with test data
y_pred = optimized_model.predict(X_test_scaled)

# Evaluate performance
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Optimized Model Accuracy: {accuracy:.2f}")
print("\nClassification Report:")
print(report)

Fitting 3 folds for each of 72 candidates, totalling 216 fits
Best hyperparameters found: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
Optimized Model Accuracy: 0.71

Classification Report:
              precision    recall  f1-score   support

       False       0.72      0.69      0.70       861
        True       0.71      0.74      0.72       878

    accuracy                           0.71      1739
   macro avg       0.71      0.71      0.71      1739
weighted avg       0.71      0.71      0.71      1739

